<a href="https://colab.research.google.com/github/davidlealo/practicos_sisrec_2026/blob/main/estudio_m%C3%A9tricas_sistemas_recomendacion.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### 1. Métricas de Error de Predicción (Ratings Explícitos)

Estas métricas evalúan qué tan cerca estuvo la "nota" que tu sistema predijo frente a la calificación real que el usuario le dio al ítem.

#### Root Mean Squared Error (RMSE)
Es el estándar de la industria para predecir ratings continuos. Al elevar el error al cuadrado, penaliza fuertemente las predicciones que están muy alejadas de la realidad.

$$RMSE = \sqrt{\frac{1}{N}\sum_{i=1}^{N}(\hat{y}_i - y_i)^2}$$

**Implementación en Python:**
```python
import numpy as np

def rmse(y_true, y_pred):
    """
    y_true: array con los ratings reales
    y_pred: array con los ratings predichos
    """
    y_true = np.array(y_true)
    y_pred = np.array(y_pred)
    
    # Calculamos la raíz de la media de los errores al cuadrado
    return np.sqrt(np.mean((y_true - y_pred)**2))
```

---

### 2. Métricas de Clasificación y Ranking (Top-K)

Estas son fundamentales cuando el objetivo es presentar al usuario una lista con los mejores $K$ ítems, sin importar el rating exacto, sino el orden.

#### Precision@K y Recall@K
Miden qué proporción de los ítems recomendados eran relevantes (Precisión) y qué proporción del total de ítems relevantes logramos capturar en nuestra recomendación (Recall).

* $Precision@K = \frac{|\text{Relevantes} \cap \text{Recomendados en Top } K|}{K}$
* $Recall@K = \frac{|\text{Relevantes} \cap \text{Recomendados en Top } K|}{|\text{Total de Relevantes}|}$

**Implementación en Python:**
```python
def precision_recall_at_k(recommended_items, relevant_items, k=10):
    """
    recommended_items: lista de IDs recomendados ordenados por probabilidad (mayor a menor)
    relevant_items: set o lista de IDs que el usuario realmente consumió
    """
    # Cortamos la lista a los top K
    top_k_recommended = recommended_items[:k]
    
    # Intersección: cuántos de los recomendados eran realmente relevantes
    hits = set(top_k_recommended).intersection(set(relevant_items))
    
    precision = len(hits) / k
    
    # Prevenir división por cero si el usuario en la prueba no tiene ítems relevantes
    recall = len(hits) / len(relevant_items) if len(relevant_items) > 0 else 0.0
    
    return precision, recall
```

#### Normalized Discounted Cumulative Gain (NDCG@K)
Esta es una métrica crucial para evaluar rankings puros. A diferencia de Precision y Recall, el NDCG asume que los aciertos en las primeras posiciones valen mucho más que los aciertos al final de la lista.

El Discounted Cumulative Gain (DCG) se calcula sumando la relevancia de cada ítem, pero "descontada" logarítmicamente según su posición $i$:

$$DCG_K = \sum_{i=1}^{K} \frac{rel_i}{\log_2(i + 1)}$$

El NDCG es simplemente el DCG dividido por el DCG Ideal (IDCG), que es el puntaje máximo posible si el modelo hubiera ordenado los ítems perfectamente de mayor a menor relevancia.

**Implementación en Python:**
```python
def dcg_at_k(relevances, k):
    """
    relevances: array de relevancias (ej. [1, 0, 1, 0, 0]) en el orden predicho por el modelo
    """
    relevances = np.asfarray(relevances)[:k]
    if relevances.size == 0:
        return 0.0
        
    # log2(i+1) donde i es el índice. En Python el índice empieza en 0,
    # por lo que el denominador correcto es log2(i+2)
    discounts = np.log2(np.arange(2, relevances.size + 2))
    return np.sum(relevances / discounts)

def ndcg_at_k(relevances, k):
    # 1. Calcular el DCG de tu modelo
    dcg = dcg_at_k(relevances, k)
    
    # 2. Calcular el DCG ideal (ordenando los aciertos primero)
    ideal_relevances = np.sort(relevances)[::-1]
    idcg = dcg_at_k(ideal_relevances, k)
    
    if idcg == 0.0:
        return 0.0
        
    # 3. Normalizar
    return dcg / idcg
```




$RMSE = \sqrt{\frac{1}{N}\sum_{i=1}^{N}(\hat{y}_i - y_i)^2}$

In [1]:
import numpy as np

def rmse(y_true, y_pred):
  y_true = np.array(y_true)
  y_pred = np.array(y_pred)

  return np.sqrt(np.mean((y_true - y_pred)**2))

In [2]:
y_real = [5, 3, 4, 4]
y_predicho = [4, 3, 4, 5]

# Desglose mental:
# 5 vs 4 -> error de 1 -> al cuadrado es 1
# 3 vs 3 -> error de 0 -> al cuadrado es 0
# 4 vs 4 -> error de 0 -> al cuadrado es 0
# 4 vs 5 -> error de -1 -> al cuadrado es 1
# Promedio de los cuadrados: (1 + 0 + 0 + 1) / 4 = 0.5
# Raíz cuadrada de 0.5 = 0.7071...

error = rmse(y_real, y_predicho)
print(f"El RMSE es: {error:.4f}") # Debería imprimir 0.7071

El RMSE es: 0.7071


¡Perfecto! Ese formato con `$` es el ideal para que tus apuntes de Markdown se vean impecables y profesionales.

Vamos con otra métrica clásica de la misma familia (predicción de ratings), ideal para seguir soltando la mano con la vectorización en `numpy`.

### Mean Absolute Error (MAE)

A diferencia del RMSE que castiga fuertemente los errores grandes (porque eleva la diferencia al cuadrado), el MAE simplemente nos da el promedio directo de las equivocaciones. Es muy útil cuando queremos saber el margen de error "típico" de nuestro modelo en la misma escala de las notas (por ejemplo, poder decirle al negocio: *"nuestro sistema se equivoca por 0.8 estrellas en promedio"*).

La fórmula matemática es:

$MAE = \frac{1}{N}\sum_{i=1}^{N}|\hat{y}_i - y_i|$

**Donde:**
* $N$ es el número total de predicciones.
* $\hat{y}_i$ es el rating predicho por tu sistema.
* $y_i$ es el rating real que dio el usuario.
* $| ... |$ representa el valor absoluto (para que los errores negativos y positivos no se cancelen entre sí).

**Tu misión:**
Escribe la función `def mae(y_true, y_pred):` asumiendo que vas a recibir listas estándar de Python y necesitas hacer el cálculo usando las bondades de `numpy`.

¡Pega tu código aquí cuando lo tengas y lo revisamos!

In [3]:
def mae(y_true, y_pred):
  y_true = np.array(y_true)
  y_pred = np.array(y_pred)

  return np.mean(np.abs(y_true - y_pred))

In [4]:
y_real = [5, 3, 4, 4]
y_predicho = [4, 3, 4, 5]

# Desglose mental para el MAE:
# 5 vs 4 -> error de 1 -> absoluto es 1
# 3 vs 3 -> error de 0 -> absoluto es 0
# 4 vs 4 -> error de 0 -> absoluto es 0
# 4 vs 5 -> error de -1 -> absoluto es 1
# Promedio de los absolutos: (1 + 0 + 0 + 1) / 4 = 0.5

error = mae(y_real, y_predicho)
print(f"El MAE es: {error}") # Imprime 0.5

El MAE es: 0.5
